In this notebook I walk through example subjects (chemistry & specialist math) of training a sentence transformer model to get an idea of what kind of models might be successful/perform the best in general, and the optimal training procedures, pretrained model to use etc.

In [2]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from InstructorEmbedding import INSTRUCTOR

from src.paths import DATA_DIR
from Qlassifier.prediction import prepare_dataframes#, get_mcq_predictions
from src.Qlassifier.evaluation import correct_in_top3

In [4]:
model = SentenceTransformer("intfloat/e5-base-v2") # https://huggingface.co/intfloat/e5-base-v2
model2 = INSTRUCTOR('hkunlp/instructor-large') # https://huggingface.co/hkunlp/instructor-large

No sentence-transformers model found with name hkunlp/instructor-large. Creating a new one with mean pooling.


We try two models: one without explicit instructions, and one with. Instructions typically help with domain-specific NLP tasks, like this one. 

In [188]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

from src.preprocessor.tree_preprocessor import TreePreprocessor, std_str
from src.Qlassifier.baseline import load_data


def get_tf_predictions(
    qn_df: pd.DataFrame,
    sd_df: pd.DataFrame,
    labels: list[int],
    model: SentenceTransformer,
    subject: str = "",
    instruct: bool = False,
) -> tuple[pd.DataFrame, np.array, np.array]:
    """ Given a model, outputs prediction on """
    qn_input = qn_df["text"] 
    report_input = qn_df["comments"]
    sd_input = sd_df["label"].str.cat(sd_df["text"], sep="")

    if instruct:
        if not subject: 
            raise ValueError("If calling with instruct==True, must have subject non-empty.")

        sd_instruct = f"Represent this {std_str(subject)} topic:"
        report_instruct = f"Represent this {std_str(subject)} examination report comment:"
        qn_instruct = f"Represent this {std_str(subject)} question"

        sd_emb = model.encode(
            [[sd_instruct, input] for input in sd_input],
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        report_emb = model.encode(
            [[report_instruct, input] for input in report_input],
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        qn_emb = model.encode(
            [[qn_instruct, input] for input in qn_input],
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    else: 
        sd_emb = model.encode(sd_input, convert_to_tensor=True, normalize_embeddings=True)
        report_emb = model.encode(report_input, convert_to_tensor=True, normalize_embeddings=True)
        qn_emb = model.encode(qn_input, convert_to_tensor=True, normalize_embeddings=True)

    cos_qn = util.cos_sim(qn_emb, sd_emb) # Embedding similarity b/w actual exam questions and study design
    best_idxs = [torch.argmax(sims).item() for sims in cos_qn]

    cos_rp = util.cos_sim(report_emb, sd_emb) # Embedding similarity b/w REPORT section on the question and study design
    best_rp_idxs = [torch.argmax(sims).item() for sims in cos_rp]

    qn_df["pred_topic"] = sd_df.loc[best_idxs, "label"].reset_index(drop=True)
    qn_df["pred_topic_idx"] = best_idxs # exam's prediction
    qn_df["report_pred"] = best_rp_idxs # report's prediction

    pred_df = qn_df.loc[:(len(labels)-1), :].copy()
    pred_df["true_topic_idx"] = labels
    pred_df["true_topic"] = sd_df.loc[labels, "label"].reset_index(drop=True)
    return pred_df, cos_qn, cos_rp

# Scope

In [64]:
subjects = [
    "chemistry",
    "specialist_mathematics"
]
subject1 = subjects[0]
chem = "2023"
chem_dir = DATA_DIR / subject1 / "past_exams"
chem_path = chem_dir / f"{chem}.pdf"

subject2 = subjects[1]
math = "2023_2"
math_dir = DATA_DIR / subject2 / "past_exams"
math_path = math_dir / f"{math}.pdf"

# Sentence Transformer

## Chem

In [101]:
qn_df, sd_df = prepare_dataframes(chem_path)

/home/ykip10/projects/Qlassifier/src/Qlassifier/fitting.py:61: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~(sd_df["label"].str.contains(


In [7]:
sd_df.head(5)

,label,text
0,Carbon-based fuels,"the definition of a fuel, including the disti..."
1,Measuring changes in chemical reactions,calculations related to the application of st...
2,Primary galvanic cells and fuel cells as sourc...,redox reactions as simultaneous oxidation and...
3,Rates of chemical reactions,factors affecting the frequency and success o...
4,Extent of chemical reactions,the distinction between reversible and irreve...


In [8]:
qn_df.head(5)

,label,text,comments,total_marks
0,Question 1,Which one of the following correctly represent...,"In human cells, glucose reacts with oxygen in ...",1
1,Question 2,Which one of the following statements is corre...,Fuel cells and galvanic cells both produce hea...,1
2,Question 3,"Molecules X, and all have the same number of c...",The larger the number of C=C double bonds in t...,1
3,Question 4,cell can be considered secondary cell if A. it...,The polarity of the physical electrodes does n...,1
4,Question 5,Consider the following statements about coenzy...,All three statements are properties of coenzym...,1


### Fitting & Evaluation

In [211]:
true_topic_mcq = [
    0, 2, 6, 5, 10, 9, 0, 4, 0, 4,
    3, 3, 6, 4, 0, 9, 1, 9, 6, 8,
    10 ,2, 6, 2, 5, 11, 12, 2, 9, 2,
]

true_topic_indices = true_topic_mcq + [
    6, 7, 7, 6, 6, 2, 2, 2, 2,   # Q1
    6, 10, 10, 10, 6, 10, 6, 10, # Q2
    0, 0, 0, 0, 0, 6,            # Q3
    7, 6, 7, 7, 7,               # Q4
    0, 4, 4, 3,                  # Q5
    2, 2, 2, 2, 2,               # Q6 
    9, 9, 9, 9, 9,               # Q7
    3, 12, 12, 1, 1, 11, 11, 11, # Q8
    5, 5                         # Q9
]

In [212]:
pred_df, cos_qn, cos_rp  = get_tf_predictions(
    qn_df, sd_df, labels=true_topic_indices, model=model
)
pred_df.head(5)

,label,text,comments,total_marks,pred_topic,pred_topic_idx,report_pred,true_topic_idx,true_topic
0,Question 1,Which one of the following correctly represent...,"In human cells, glucose reacts with oxygen in ...",1,Carbon-based fuels,0,0,0,Carbon-based fuels
1,Question 2,Which one of the following statements is corre...,Fuel cells and galvanic cells both produce hea...,1,Primary galvanic cells and fuel cells as sourc...,2,2,2,Primary galvanic cells and fuel cells as sourc...
2,Question 3,"Molecules X, and all have the same number of c...",The larger the number of C=C double bonds in t...,1,"Structure, nomenclature and properties of orga...",6,6,6,"Structure, nomenclature and properties of orga..."
3,Question 4,cell can be considered secondary cell if A. it...,The polarity of the physical electrodes does n...,1,Primary galvanic cells and fuel cells as sourc...,2,2,5,Production of chemicals using electrolysis
4,Question 5,Consider the following statements about coenzy...,All three statements are properties of coenzym...,1,Reactions of organic compounds,7,7,10,Medicinal chemistry


In [213]:
def evaluate_model(df, cos):
    n = len(df)
    ex_correct_pc = len(df[df["true_topic_idx"] == df["pred_topic_idx"]]) / n
    rp_correct_pc = len(df[df["true_topic_idx"] == df["report_pred"]]) / n

    one_correct_pc = len(
       df[(df["true_topic_idx"] == df["pred_topic_idx"]) |
              (df["true_topic_idx"] == df["report_pred"])]
       ) / n  

    print(f"Exam-only accuracy: {100*ex_correct_pc:.2f}%")
    print(f"Report-only accuracy: {100*rp_correct_pc:.2f}%")
    print(f"The percent of time either the report or the exam labels a question correctly is {100*one_correct_pc:.2f}%")
    
    print("The percentage of time the correct topic is among the top 3 predicted topics is "
         f"{100*(sum([correct_in_top3(qn_idx,df,cos) for qn_idx in range(n)]) / n):.2f}%")

evaluate_model(pred_df, cos_qn)

Exam-only accuracy: 43.90%
Report-only accuracy: 48.78%
The percent of time either the report or the exam labels a question correctly is 67.07%
The percentage of time the correct topic is among the top 3 predicted topics is 75.61%


Perhaps we can combine the model's predictions using the report and question in some way to models accuracy.

It should be noted that combining exam and report into one query to the model has been tried, but does not lead to increases in performance.

### Fine-tuning with SimCSE

SimCSE is an unsupervised learning algorithm that works by taking the same sentence twice, introducing dropout noise, then, based on the fact that these two features should be close together in embedding space, optimises a loss. 

We should see if we can improve the model by fine tuning it.

In [74]:
from sentence_transformers import losses
from torch.utils.data import DataLoader
from sentence_transformers import InputExample

In [86]:
training_exam_dfs = []
training_report_dfs = []

for exam_path in chem_dir.iterdir():
    if chem in str(exam_path):
        # leave current dataset for evaluation
        continue
    curr_exam = exam_path.stem
    train_exam_df, train_report_df, _ = prepare_dataframes(exam_path)
    train_exam_df.insert(0, "Exam Year", [curr_exam]*len(train_exam_df))
    train_report_df.insert(0, "Exam Year", [curr_exam]*len(train_report_df))

    training_exam_dfs.append(train_exam_df)
    training_report_dfs.append(train_report_df)

KeyboardInterrupt: 

In [ ]:
# ["Exam Year", "Label"] unique for each year
training_ex_df = pd.concat(training_exam_dfs) \
                   .reset_index(drop=True) \
                   .drop_duplicates(subset=["Exam Year", "label"])

training_rp_df = pd.concat(training_report_dfs) \
                   .reset_index(drop=True) \
                   .rename({"comments": "text"}, axis=1) \
                   .fillna(value="")

ex_train_examples = [InputExample(texts=[s, s]) for s in training_ex_df["text"]]
rp_train_examples = [InputExample(texts=[s, s]) for s in training_rp_df["text"]]

ex_dataloader = DataLoader(ex_train_examples, shuffle=True, batch_size=32)

loss = losses.MultipleNegativesRankingLoss(model)
model.fit(train_objectives=[(ex_dataloader, loss)], epochs=1)

Step,Training Loss


In [ ]:
preds = get_mcq_predictions()

,label,text,input,predicted_topic,predicted_topic_idx,report_predict,true_topic_idx,true_topic
0,Question 1,Which one of the following correctly represent...,Which one of the following correctly represent...,Carbon-based fuels,0,0,0,Carbon-based fuels
1,Question 2,Which one of the following statements is corre...,Which one of the following statements is corre...,Primary galvanic cells and fuel cells as sourc...,2,2,2,Primary galvanic cells and fuel cells as sourc...
2,Question 3,"Molecules X, and all have the same number of c...","Molecules X, and all have the same number of c...","Structure, nomenclature and properties of orga...",6,6,6,"Structure, nomenclature and properties of orga..."
3,Question 4,cell can be considered secondary cell if A. it...,cell can be considered secondary cell if A. it...,Primary galvanic cells and fuel cells as sourc...,2,2,5,Production of chemicals using electrolysis
4,Question 5,Consider the following statements about coenzy...,Consider the following statements about coenzy...,Reactions of organic compounds,7,7,10,Medicinal chemistry
5,Question 6,Consider the following statements. I. HPLC is ...,Consider the following statements. I. HPLC is ...,Laboratory analysis of organic compounds,8,9,9,Instrumental analysis of organic compounds
6,Question 7,Consider the following statements about fossil...,Consider the following statements about fossil...,Carbon-based fuels,0,0,0,Carbon-based fuels
7,Question 8,"At 327 °C, the equilibrium constant for the re...","At 327 °C, the equilibrium constant for the re...",Extent of chemical reactions,4,4,4,Extent of chemical reactions
8,Question 9,Consider the following statements about coal s...,Consider the following statements about coal s...,Carbon-based fuels,0,0,0,Carbon-based fuels
9,Question 10,"At constant temperature, which one of the foll...","At constant temperature, which one of the foll...",Extent of chemical reactions,4,4,4,Extent of chemical reactions


The accuracy here is about 66% (I forgot to compute in python, I closed my IDE, do not want to re-train model).

Yeah so clearly our approach is wrong. It doesn't make sense to train the model to look for semantic similarities using SimCSE because two questions belonging to two different topics can be semantically similar (Such as "Calculate the ... of ..."), so an embedding space which doesn't bring together two similar topics doesn't help us.  

Next, I want to check how the model works on the specialist mathematics exams. Without customising, for the same reason as above.

## Specialist Math

Let's just use the same model and approach. We learnt that there's really no benefit to fine-tuning (on this model, for this task, with our data), so we won't waste time doing that. 

In [203]:
mcq_labels = [
    0, 1, 1, 2, 2, 4, 4, 4, 7, 3, 
    3, 5, 5, 6, 6, 8, 6, 7, 9, 11,
]

ground_truth = mcq_labels + [
    1, 3, 3, 3, 7, 8, 3, 3,            # Q1
    2, 2, 2, 2, 2, 2, 2, 2,            # Q2
    3, 3, 3, 3, 3, 3,                  # Q3
    4, 4, 4, 4, 4, 4, 1, 4,            # Q4
    6, 6, 7, 7, 7, 7,                  # Q5
    11, 11, 11, 12, 12, 12, 12, 12, 10 # Q6  

]

qn_df_math, sd_df_math = prepare_dataframes(math_path)
mcq_df_math, cos_qn_math, cos_rp_math  = get_mcq_predictions(
    qn_df_math, sd_df_math, labels=ground_truth, model=model
)

In [191]:
evaluate_model(mcq_df_math, cos_qn_math)

Exam-only accuracy: 50.77%
Report-only accuracy: 18.46%
The percent of time either the report or the exam labels a question correctly is 55.38%
The percentage of time the correct topic is among the top 3 predicted topics is 80.0%


The report comments are mostly empty for these mcq questions. We note that if this is the case, the report's predictions should be weighed very lightly. For the chemistry case, where the report text is dense, the reports seem to capture some patterns uncaptured by the exam questions.

Also, we learnt that this model actually performs substantially better in the mathematics (at least on this limited, small mcq only dataset) when compared to the TF-IDF approach (although still not great).

# Instructor

## Chem

In [195]:
mcq_df, cos_qn, cos_rp  = get_mcq_predictions(
    qn_df, sd_df, labels=true_topic_indices, model=model2, subject="Chemistry", instruct=True
)
mcq_df.head(5)

`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


,label,text,comments,total_marks,pred_topic,pred_topic_idx,report_pred,true_topic_idx,true_topic
0,Question 1,Which one of the following correctly represent...,"In human cells, glucose reacts with oxygen in ...",1,Carbon-based fuels,0,0,0,Carbon-based fuels
1,Question 2,Which one of the following statements is corre...,Fuel cells and galvanic cells both produce hea...,1,Primary galvanic cells and fuel cells as sourc...,2,2,2,Primary galvanic cells and fuel cells as sourc...
2,Question 3,"Molecules X, and all have the same number of c...",The larger the number of C=C double bonds in t...,1,"Structure, nomenclature and properties of orga...",6,6,6,"Structure, nomenclature and properties of orga..."
3,Question 4,cell can be considered secondary cell if A. it...,The polarity of the physical electrodes does n...,1,Primary galvanic cells and fuel cells as sourc...,2,2,5,Production of chemicals using electrolysis
4,Question 5,Consider the following statements about coenzy...,All three statements are properties of coenzym...,1,Rates of chemical reactions,3,3,10,Medicinal chemistry


In [148]:
evaluate_model(mcq_df, cos_qn)

Exam-only accuracy: 66.67%
Report-only accuracy: 63.33%
The percent of time either the report or the exam labels a question correctly is 73.33%
The percentage of time the correct topic is among the top 3 predicted topics is 93.33333333333333%


## Math

In [205]:
mcq_df_math, cos_qn_math, cos_rp_math  = get_mcq_predictions(
    qn_df_math, sd_df_math, labels=ground_truth, model=model2, subject="Mathematics", instruct=True
)

`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


In [206]:
evaluate_model(mcq_df_math, cos_qn_math)

Exam-only accuracy: 81.54%
Report-only accuracy: 18.46%
The percent of time either the report or the exam labels a question correctly is 83.08%
The percentage of time the correct topic is among the top 3 predicted topics is 93.84615384615384%


The instructor model performs far better for mathematics than the purely sentence-transformer one.